# Track B: dataset profiling for Chapter 3, Section 3.3.2

Profiles the seven external datasets so the report only names data that has been loaded and checked.

**No GPU needed.** Everything here is downloading files and counting rows, which runs on CPU. A standard Colab runtime is enough.

**Before you start**
1. Kaggle API token: kaggle.com > Settings > API > *Create New Token*. This downloads `kaggle.json`. You will upload it in Step 2.
2. Open each Kaggle dataset page once while logged in and accept its rules if a button asks you to. Otherwise that download fails with a 403.

**Output:** `dataset_profile_report.md` and `dataset_profile_report.json`, downloaded automatically at the end.

**Privacy:** for the resume datasets the notebook reports *counts* of email and phone patterns only. It never prints them, and previews are masked.

## Step 1: install

In [1]:
!pip -q install datasets kaggle pypdf pandas pyarrow openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 12.1 MB/s eta 0:00:00


## Step 2: Kaggle credentials (upload `kaggle.json` when prompted)

In [3]:
import os, json
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(exist_ok=True)
if not (kaggle_dir / "kaggle.json").exists():
    from google.colab import files
    print("Upload kaggle.json")
    up = files.upload()
    (kaggle_dir / "kaggle.json").write_bytes(up["kaggle.json"])
os.chmod(kaggle_dir / "kaggle.json", 0o600)
print("Kaggle token ready")

Upload kaggle.json


Saving kaggle.json to kaggle.json
Kaggle token ready


## Step 3: configuration

In [4]:
import datetime as dt

TODAY = dt.date.today().isoformat()
DATA = Path("/content/datasets")
DATA.mkdir(exist_ok=True)

DATASETS = [
    {"id": "roman_urdu_parl", "name": "Roman-Urdu-Parl (HF split)", "source": "hf",
     "ref": "Mavkif/Roman-Urdu-Parl-split", "role": "Multilingual benchmark source"},
    {"id": "linkedin", "name": "LinkedIn-Dataset (Ayoobi et al.)", "source": "github",
     "ref": "https://github.com/navid-aub/LinkedIn-Dataset", "role": "Profile-verification reference"},
    {"id": "crunchbase_success", "name": "Kaggle: Startup Success/Fail (Crunchbase)", "source": "kaggle",
     "ref": "yanmaksi/big-startup-secsees-fail-dataset-from-crunchbase", "role": "Schema proxy only"},
    {"id": "crunchbase_invest", "name": "Kaggle: StartUp Investments (Crunchbase)", "source": "kaggle",
     "ref": "arindam235/startup-investments-crunchbase", "role": "Schema proxy only"},
    {"id": "hse", "name": "HSE Resume-Job Matching", "source": "kaggle",
     "ref": "darysha/hse-hackathon", "role": "Matching-pipeline practice proxy"},
    {"id": "resume_pdf", "name": "Resume Dataset PDF", "source": "kaggle",
     "ref": "hadikp/resume-data-pdf", "role": "Test inputs for resume optimizer"},
    {"id": "ats", "name": "ATS Dataset NLP+ML", "source": "kaggle",
     "ref": "amaymishra11/resume-screening-dataset-ats-and-nlp-ready",
     "role": "Resume/JD text pairs as test inputs (scores ignored)"},
]

# Datasets whose text may contain personal data: mask previews, count PII patterns only.
PII_SENSITIVE = {"linkedin", "hse", "resume_pdf", "ats"}

RESULTS = {}

## Step 4: helper functions

In [9]:
import re, subprocess, hashlib
import pandas as pd

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")
PHONE = re.compile(r"(?:\+?\d[\d\s().-]{8,}\d)")

def mask(text):
    text = EMAIL.sub("[EMAIL]", str(text))
    return PHONE.sub("[PHONE]", text)

def pii_counts(series_list):
    e = p = 0
    for s in series_list:
        joined = " ".join(map(str, s.dropna().astype(str).head(5000)))
        e += len(EMAIL.findall(joined)); p += len(PHONE.findall(joined))
    return {"email_patterns_in_first_5000_rows": e, "phone_patterns_in_first_5000_rows": p}

def read_any(path):
    suf = path.suffix.lower()
    if suf == ".csv":
        try:
            return pd.read_csv(path, low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, low_memory=False, encoding="latin-1")
    if suf in (".xlsx", ".xls"):
        return pd.read_excel(path)
    if suf == ".parquet":
        return pd.read_parquet(path)
    if suf == ".json":
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)
    if suf == ".jsonl":
        return pd.read_json(path, lines=True)
    return None

def profile_df(df, sensitive=False):
    rec = {"rows": int(len(df)), "columns": int(df.shape[1])}
    rec["dtypes"] = {c: str(t) for c, t in df.dtypes.items()}
    miss = (df.isna().mean() * 100).round(2)
    rec["missing_pct"] = {c: float(v) for c, v in miss.items() if v > 0}
    rec["duplicate_rows"] = int(df.duplicated().sum()) if len(df) < 2_000_000 else "skipped (large)"
    # Candidate label columns: few distinct values
    labels = {}
    for c in df.columns:
        try:
            n = df[c].nunique(dropna=True)
            if 1 < n <= 15:
                labels[c] = {str(k): int(v) for k, v in df[c].value_counts(dropna=False).head(15).items()}
        except TypeError:
            # Handle unhashable types (like nested lists in JSON)
            continue
    rec["low_cardinality_columns"] = labels
    # Long-text columns: evidence that full resume / job-description text is present
    text_cols = {}
    for c in df.columns:
        if df[c].dtype == object or pd.api.types.is_string_dtype(df[c]):
            try:
                lens = df[c].dropna().astype(str).str.len()
                if len(lens) and lens.mean() > 200:
                    text_cols[c] = {"mean_chars": int(lens.mean()), "median_chars": int(lens.median())}
            except Exception:
                continue
    rec["long_text_columns"] = text_cols
    if sensitive and text_cols:
        rec["pii_check"] = pii_counts([df[c] for c in text_cols])
    sample = df.head(3).astype(str)
    if sensitive:
        sample = sample.map(lambda v: mask(v)[:150])
    else:
        sample = sample.map(lambda v: v[:150])
    rec["sample_rows"] = sample.to_dict(orient="records")
    return rec

def profile_folder(folder, sensitive=False):
    """Profile every tabular file under a folder; summarise everything else by extension."""
    out = {"files": {}, "other_files_by_extension": {}}
    for f in sorted(folder.rglob("*")):
        if not f.is_file() or ".git" in f.parts:
            continue
        # Skip dataset-metadata.json specifically to prevent indexing metadata as a dataset
        if f.name == "dataset-metadata.json":
            ext = f.suffix.lower() or "(none)"
            out["other_files_by_extension"][ext] = out["other_files_by_extension"].get(ext, 0) + 1
            continue
        df = None
        try:
            df = read_any(f)
        except Exception as e:
            out["files"][str(f.relative_to(folder))] = {"error": f"{type(e).__name__}: {e}"}
            continue
        if df is not None:
            print(f"  tabular: {f.relative_to(folder)}  {df.shape}")
            out["files"][str(f.relative_to(folder))] = profile_df(df, sensitive)
        else:
            ext = f.suffix.lower() or "(none)"
            out["other_files_by_extension"][ext] = out["other_files_by_extension"].get(ext, 0) + 1
    out["size_mb"] = round(sum(f.stat().st_size for f in folder.rglob("*") if f.is_file()) / 1_048_576, 2)
    return out

def kaggle_licence(ref, dest):
    try:
        subprocess.run(["kaggle", "datasets", "metadata", "-d", ref, "-p", str(dest)],
                       check=True, capture_output=True, text=True)
        meta = json.loads((dest / "dataset-metadata.json").read_text())
        lic = meta.get("licenses") or []
        return ", ".join(l.get("name", "?") for l in lic) or "not stated"
    except Exception as e:
        return f"could not read ({type(e).__name__})"

print("helpers ready")

helpers ready


## Step 5: Roman-Urdu-Parl (Hugging Face)
Row counts come from the dataset metadata, and the sample is streamed, so the full ~4M+ rows are never loaded into memory.

In [10]:
from datasets import load_dataset_builder, load_dataset

d = DATASETS[0]
rec = {"name": d["name"], "source": d["ref"], "role": d["role"], "access_date": TODAY}
try:
    builder = load_dataset_builder(d["ref"])
    info = builder.info
    rec["licence"] = info.license or "not stated in dataset card metadata"
    rec["features"] = {k: str(v) for k, v in (info.features or {}).items()}
    rec["splits"] = {k: int(v.num_examples) for k, v in (info.splits or {}).items()}
    rec["total_rows"] = sum(rec["splits"].values()) if rec["splits"] else "not in metadata"
    first_split = next(iter(rec["splits"])) if rec["splits"] else "train"
    stream = load_dataset(d["ref"], split=first_split, streaming=True)
    rec["sample_20_pairs"] = [dict(r) for _, r in zip(range(20), stream)]
except Exception as e:
    rec["error"] = f"{type(e).__name__}: {e}"
RESULTS[d["id"]] = rec
print(json.dumps({k: v for k, v in rec.items() if k != "sample_20_pairs"}, indent=2, ensure_ascii=False))
for row in rec.get("sample_20_pairs", [])[:5]:
    print(row)
print("\nCheck by eye: are the Urdu and Roman Urdu sides of each pair the same sentence?")

{
  "name": "Roman-Urdu-Parl (HF split)",
  "source": "Mavkif/Roman-Urdu-Parl-split",
  "role": "Multilingual benchmark source",
  "access_date": "2026-10-03",
  "licence": "not stated in dataset card metadata",
  "features": {
    "Urdu text": "Value('string')",
    "Roman-Urdu text": "Value('string')"
  },
  "splits": {
    "train": 6333218,
    "validation": 20719,
    "test": 20741
  },
  "total_rows": 6374678
}
{'Urdu text': 'وکیل', 'Roman-Urdu text': 'wakeel'}
{'Urdu text': 'وکیل', 'Roman-Urdu text': 'wakeel'}
{'Urdu text': 'وکیل ایک ایسی شخصیت کو کہا جاتا ہے کہ جو دوسرے اپنے صارف کی جانب سے یا اسکی بابت گفتگو کرے اس مضمون میں یہ گفتگو قانون سے متعلق تصور کی گئی ہے اور اس وجہ سے یہ مضمون صرف قانونی وکلا کے بارے میں ذکر کرتا ہے', 'Roman-Urdu text': 'wakeel aik aisi shakhsiyat ko kaha jata hai ke jo dosray apne Sarif ki janib se ya uski baabat guftagu kere is mazmoon mein yeh guftagu qanoon se mutaliq tasawwur ki gayi hai aur is wajah se yeh mazmoon sirf qanooni wukla ke baray mein

## Step 6: LinkedIn-Dataset (GitHub)

In [11]:
d = DATASETS[1]
dest = DATA / d["id"]
if not dest.exists():
    subprocess.run(["git", "clone", "--depth", "1", d["ref"], str(dest)], check=True)
rec = {"name": d["name"], "source": d["ref"], "role": d["role"], "access_date": TODAY}
lic_files = [f.name for f in dest.iterdir() if f.name.lower().startswith("licen")]
rec["licence"] = (dest / lic_files[0]).read_text()[:200] if lic_files else "no LICENSE file in repo"
rec.update(profile_folder(dest, sensitive=True))
RESULTS[d["id"]] = rec
print("Expected: 3,600 profiles = 1,800 legitimate + 600 fake + 1,200 ChatGPT-generated. Compare with the counts above.")

Expected: 3,600 profiles = 1,800 legitimate + 600 fake + 1,200 ChatGPT-generated. Compare with the counts above.


## Step 7: Kaggle datasets (tabular)

In [12]:
for d in DATASETS[2:]:
    if d["id"] == "resume_pdf":
        continue  # handled in Step 8
    print(f"\n=== {d['name']}")
    dest = DATA / d["id"]
    dest.mkdir(exist_ok=True)
    rec = {"name": d["name"], "source": f"https://www.kaggle.com/datasets/{d['ref']}",
           "role": d["role"], "access_date": TODAY}
    try:
        if not any(dest.iterdir()):
            subprocess.run(["kaggle", "datasets", "download", "-d", d["ref"], "-p", str(dest), "--unzip"],
                           check=True, capture_output=True, text=True)
        rec["licence"] = kaggle_licence(d["ref"], dest)
        rec.update(profile_folder(dest, sensitive=d["id"] in PII_SENSITIVE))
    except subprocess.CalledProcessError as e:
        rec["error"] = (e.stderr or str(e))[:300]
        print("  DOWNLOAD FAILED. If 403: open the dataset page on Kaggle and accept its rules, then rerun.")
    RESULTS[d["id"]] = rec


=== Kaggle: Startup Success/Fail (Crunchbase)
  tabular: big_startup_secsees_dataset.csv  (66368, 14)

=== Kaggle: StartUp Investments (Crunchbase)
  tabular: investments_VC.csv  (54294, 39)

=== HSE Resume-Job Matching

=== ATS Dataset NLP+ML
  tabular: ats_resume_dataset_elite_v3.csv  (6000, 17)


## Step 8: Resume Dataset PDF (folder of PDFs)
Counts files per category folder and tests text extraction on a sample. Image-converted PDFs often extract no text.

In [13]:
from pypdf import PdfReader
import random

d = next(x for x in DATASETS if x["id"] == "resume_pdf")
dest = DATA / d["id"]
dest.mkdir(exist_ok=True)
rec = {"name": d["name"], "source": f"https://www.kaggle.com/datasets/{d['ref']}",
       "role": d["role"], "access_date": TODAY}
try:
    if not any(dest.iterdir()):
        subprocess.run(["kaggle", "datasets", "download", "-d", d["ref"], "-p", str(dest), "--unzip"],
                       check=True, capture_output=True, text=True)
    rec["licence"] = kaggle_licence(d["ref"], dest)
    pdfs = [f for f in dest.rglob("*.pdf")]
    rec["total_pdfs"] = len(pdfs)
    per_folder = {}
    for f in pdfs:
        per_folder[f.parent.name] = per_folder.get(f.parent.name, 0) + 1
    rec["pdfs_per_folder"] = dict(sorted(per_folder.items()))
    random.seed(42)
    sample = random.sample(pdfs, min(30, len(pdfs)))
    with_text, chars, emails, phones, previews = 0, [], 0, 0, []
    for f in sample:
        try:
            txt = " ".join((p.extract_text() or "") for p in PdfReader(str(f)).pages)
        except Exception:
            txt = ""
        if len(txt.strip()) >= 100:
            with_text += 1
        chars.append(len(txt))
        emails += len(EMAIL.findall(txt)); phones += len(PHONE.findall(txt))
        if len(previews) < 3 and txt.strip():
            previews.append(mask(" ".join(txt.split()))[:200])
    rec["text_extraction_on_sample"] = {
        "sample_size": len(sample), "files_with_text": with_text,
        "pct_with_text": round(100 * with_text / max(len(sample), 1), 1),
        "median_chars": int(pd.Series(chars).median()) if chars else 0}
    rec["pii_check_on_sample"] = {"email_patterns": emails, "phone_patterns": phones}
    rec["masked_previews"] = previews
    other = {}
    for f in dest.rglob("*"):
        if f.is_file() and f.suffix.lower() != ".pdf":
            other[f.suffix.lower() or "(none)"] = other.get(f.suffix.lower() or "(none)", 0) + 1
    rec["non_pdf_files"] = other
except subprocess.CalledProcessError as e:
    rec["error"] = (e.stderr or str(e))[:300]
RESULTS[d["id"]] = rec
print(json.dumps({k: v for k, v in rec.items() if k != "masked_previews"}, indent=2))
print("\nIf email/phone counts are high, the resumes are real and unredacted: that dataset drops out.")

{
  "name": "Resume Dataset PDF",
  "source": "https://www.kaggle.com/datasets/hadikp/resume-data-pdf",
  "role": "Test inputs for resume optimizer",
  "access_date": "2026-10-03",
  "licence": "not stated",
  "total_pdfs": 8905,
  "pdfs_per_folder": {
    "Accountant": 284,
    "Accountant resumes": 67,
    "Advocate": 247,
    "Advocate resumes": 94,
    "Agricultural": 199,
    "Agricultural resumes": 68,
    "Agriculture": 79,
    "Apparel": 69,
    "Apparel resumes": 51,
    "Architect": 84,
    "Architects resumes": 60,
    "Arts": 255,
    "Arts resumes": 68,
    "Automobile": 73,
    "Automobile resumes": 40,
    "Aviation": 200,
    "Aviation resumes": 51,
    "BPO resumes": 29,
    "Banking": 221,
    "Banking resumes": 41,
    "Blockchain": 24,
    "Blockchain resumes": 8,
    "Building _Construction resumes": 55,
    "Business Analyst resumes": 56,
    "BusinessAnalyst": 77,
    "Civil Engineer resumes": 73,
    "CivilEngineer": 76,
    "Consult": 99,
    "Consultant": 148,

## Step 9: write the report

In [14]:
def fmt_files(files):
    lines = []
    for fname, r in files.items():
        if "error" in r:
            lines.append(f"- `{fname}`: ERROR {r['error']}")
            continue
        lines.append(f"- `{fname}`: {r['rows']:,} rows x {r['columns']} columns, duplicates: {r['duplicate_rows']}")
        if r["missing_pct"]:
            top = sorted(r["missing_pct"].items(), key=lambda x: -x[1])[:8]
            lines.append("  - Missing (%): " + ", ".join(f"{c} {v}" for c, v in top))
        if r["low_cardinality_columns"]:
            for c, vc in list(r["low_cardinality_columns"].items())[:5]:
                lines.append(f"  - `{c}` values: {vc}")
        if r["long_text_columns"]:
            lines.append(f"  - Long-text columns: {r['long_text_columns']}")
        if "pii_check" in r:
            lines.append(f"  - PII pattern counts: {r['pii_check']}")
        lines.append(f"  - Columns: {', '.join(list(r['dtypes'])[:25])}")
    return lines

out = [f"# Dataset profiling report (Track B)", "", f"Run date: {TODAY}", ""]
for d in DATASETS:
    r = RESULTS.get(d["id"], {"error": "not run"})
    out += [f"## {d['name']}", "", f"- Source: {r.get('source', d['ref'])}",
            f"- Role: {d['role']}", f"- Access date: {r.get('access_date', TODAY)}",
            f"- Licence: {r.get('licence', 'not checked')}"]
    if "error" in r:
        out += [f"- ERROR: {r['error']}", ""]
        continue
    if d["id"] == "roman_urdu_parl":
        out += [f"- Splits: {r.get('splits')}", f"- Total rows: {r.get('total_rows')}",
                f"- Features: {r.get('features')}", "- First 5 sampled pairs:"]
        out += [f"  - {json.dumps(s, ensure_ascii=False)[:250]}" for s in r.get("sample_20_pairs", [])[:5]]
    elif d["id"] == "resume_pdf":
        out += [f"- Total PDFs: {r.get('total_pdfs')}", f"- Per folder: {r.get('pdfs_per_folder')}",
                f"- Text extraction (sample): {r.get('text_extraction_on_sample')}",
                f"- PII patterns (sample): {r.get('pii_check_on_sample')}",
                f"- Non-PDF files: {r.get('non_pdf_files')}"]
    else:
        out += [f"- Size: {r.get('size_mb')} MB", f"- Other files: {r.get('other_files_by_extension')}"]
        out += fmt_files(r.get("files", {}))
    out.append("")

Path("dataset_profile_report.md").write_text("\n".join(out), encoding="utf-8")
Path("dataset_profile_report.json").write_text(json.dumps(RESULTS, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print("\n".join(out)[:4000])

# Dataset profiling report (Track B)

Run date: 2026-10-03

## Roman-Urdu-Parl (HF split)

- Source: Mavkif/Roman-Urdu-Parl-split
- Role: Multilingual benchmark source
- Access date: 2026-10-03
- Licence: not stated in dataset card metadata
- Splits: {'train': 6333218, 'validation': 20719, 'test': 20741}
- Total rows: 6374678
- Features: {'Urdu text': "Value('string')", 'Roman-Urdu text': "Value('string')"}
- First 5 sampled pairs:
  - {"Urdu text": "وکیل", "Roman-Urdu text": "wakeel"}
  - {"Urdu text": "وکیل", "Roman-Urdu text": "wakeel"}
  - {"Urdu text": "وکیل ایک ایسی شخصیت کو کہا جاتا ہے کہ جو دوسرے اپنے صارف کی جانب سے یا اسکی بابت گفتگو کرے اس مضمون میں یہ گفتگو قانون سے متعلق تصور کی گئی ہے اور اس وجہ سے یہ مضمون صرف قانونی وکلا کے بارے میں ذکر کرتا ہے", "Roman-Urdu text": "wakeel 
  - {"Urdu text": "وکیل قانون", "Roman-Urdu text": "wakeel qanoon"}
  - {"Urdu text": "وکیل قانون ایک شخص جسے دوسرے شخص کی جگہ کام کرنے یا کی نمائندگی کرنے کا اختیار حاصل ہوتا ہے وکیل سفر ایک شخص جو 

## Step 10: download the report

In [15]:
from google.colab import files
files.download("dataset_profile_report.md")
files.download("dataset_profile_report.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
f = next((DATA/"linkedin").rglob("*.pcl"))
obj = pd.read_pickle(f)
print(type(obj))
if isinstance(obj, pd.DataFrame):
    print(obj.shape); print(obj.columns.tolist()); print(obj.iloc[:, -1].value_counts().head(10))
elif isinstance(obj, dict):
    print({k: (type(v).__name__, len(v) if hasattr(v, "__len__") else "") for k, v in list(obj.items())[:10]})

<class 'pandas.core.frame.DataFrame'>
(3600, 39)
['Intro', 'Full Name', 'Workplace', 'Location', 'Connections', 'Photo', 'Followers', 'About', 'Experiences', 'Number of Experiences', 'Educations', 'Number of Educations', 'Licenses', 'Number of Licenses', 'Volunteering', 'Number of Volunteering', 'Skills', 'Number of Skills', 'Recommendations', 'Number of Recommendations', 'Projects', 'Number of Projects', 'Publications', 'Number of Publications', 'Courses', 'Number of Courses', 'Honors', 'Number of Honors', 'Scores', 'Number of Scores', 'Languages', 'Number of Languages', 'Organizations', 'Number of Organizations', 'Interests', 'Number of Interests', 'Activities', 'Number of Activities', 'Label']
Label
0     1800
1      600
10     600
11     600
Name: count, dtype: int64


In [17]:
p = DATA/"hse"/"train.csv"
print(open(p, encoding="utf-8", errors="replace").read(1500))
df = pd.read_csv(p, sep=None, engine="python", nrows=5000, on_bad_lines="warn")
print(df.shape); print(df.columns.tolist())
for c in df.columns:
    if df[c].nunique() <= 15: print(c, df[c].value_counts().to_dict())

idCv|idVacancy|cv_status|locality|localityName|birthday|gender|age|positionName|dateCreate|typicalPosition_cv|dateModify_cv|publishedDate|academicDegree|skills_cv|driveLicenses|experience|otherCertificates|country|educationList|hardSkills_cv|softSkills_cv|workExperienceList|scheduleType_cv|salary_cv|retrainingCapability_cv|businessTrip|languageKnowledge_cv|relocation|innerInfo|salaryMin_cv|salaryMax_cv|education|id_vacancy|stateRegionCode_vacancy|vacancyName|codeProfession|codeProfessionalSphere|professionalSphereName|vacancyAddress|vacancyAddressHouse|vacancyAddressAdditionalInfo|geo|salary_vacancy|languageKnowledge_vacancy|busyType_vacancy|educationRequirements|hardSkills_vacancy|softSkills_vacancy|skills_vacancy|typicalPosition_vacancy|experienceRequirements|scheduleType_vacancy|otherVacancyBenefit|careerPerspective|idPriorityCategory|needMedcard|sourceType|requiredDriveLicense|retrainingCapability_vacancy|transportCompensation|changeTime|contactPerson|contactSource|company|fullComp

Streaming output truncated to the last 5000 lines.

  df = pd.read_csv(p, sep=None, engine="python", nrows=5000, on_bad_lines="warn")
/tmp/ipykernel_11744/2926907261.py:3: ParserWarning: Skipping line 148829: 'e' expected after '"'

  df = pd.read_csv(p, sep=None, engine="python", nrows=5000, on_bad_lines="warn")
/tmp/ipykernel_11744/2926907261.py:3: ParserWarning: Skipping line 148830: 'e' expected after '"'

  df = pd.read_csv(p, sep=None, engine="python", nrows=5000, on_bad_lines="warn")
/tmp/ipykernel_11744/2926907261.py:3: ParserWarning: Skipping line 148831: 'e' expected after '"'

  df = pd.read_csv(p, sep=None, engine="python", nrows=5000, on_bad_lines="warn")
/tmp/ipykernel_11744/2926907261.py:3: ParserWarning: Skipping line 148832: 'e' expected after '"'

  df = pd.read_csv(p, sep=None, engine="python", nrows=5000, on_bad_lines="warn")
/tmp/ipykernel_11744/2926907261.py:3: ParserWarning: Skipping line 148833: 'e' expected after '"'

  df = pd.read_csv(p, sep=None, engine="pyt

(0, 141)
['idCv|idVacancy|cv_status|locality|localityNam', '|birthday|g', 'nd', 'r|ag', '|positionNam', '|dat', 'Cr', 'at', '|typicalPosition_cv|dat', 'Modify_cv|publish', 'dDat', '|acad', 'micD', 'gr', 'Unnamed: 14', '|skills_cv|driv', 'Lic', 'ns', 's|', 'xp', 'ri', 'nc', '|oth', 'rC', 'rtificat', 's|country|', 'ducationList|hardSkills_cv|softSkills_cv|workExp', 'ri.1', 'nc.1', 'List|sch', 'dul', 'Typ', '_cv|salary_cv|r', 'trainingCapability_cv|busin', 'ssTrip|languag', 'Knowl', 'dg', '_cv|r', 'location|inn', 'rInfo|salaryMin_cv|salaryMax_cv|', 'ducation|id_vacancy|stat', 'R', 'gionCod', '_vacancy|vacancyNam', '|cod', 'Prof', 'ssion|cod', 'Prof.1', 'ssionalSph', 'r', '|prof', 'ssionalSph.1', 'r.1', 'Nam', '|vacancyAddr', 'ss|vacancyAddr', 'ssHous', '|vacancyAddr.1', 'ssAdditionalInfo|g', 'o|salary_vacancy|languag', 'Knowl.1', 'dg.1', '_vacancy|busyTyp', '_vacancy|', 'ducationR', 'quir', 'm', 'nts|hardSkills_vacancy|softSkills_vacancy|skills_vacancy|typicalPosition_vacancy|', 'xp.1', '

In [18]:
import warnings; warnings.filterwarnings("ignore")
df = pd.read_csv(p, sep="|", nrows=200_000, on_bad_lines="skip", low_memory=False)
print(df.shape)
print(df["cv_status"].value_counts(dropna=False))
print(df[["idCv", "idVacancy"]].nunique())
n = sum(len(c) for c in pd.read_csv(p, sep="|", usecols=["cv_status"], chunksize=500_000, on_bad_lines="skip"))
print("total rows:", n)

(150493, 91)
cv_status
Отказ          129503
Приглашение     20990
Name: count, dtype: int64
idCv         65714
idVacancy    43756
dtype: int64
total rows: 150493
